In [2]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def scrape_movie_chart_requests():
    folder_name = "movie_images"
    os.makedirs(folder_name, exist_ok=True)

    url = "https://www.moviechart.co.kr/rank/realtime/index/image"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    try:
        print("페이지 정보를 가져오는 중입니다...")
        response = requests.get(url, headers=headers)
        response.raise_for_status() 
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 전체 리스트 항목 가져오기
        movie_list = soup.select('.listTable tbody tr') or soup.select('.movie-box') or soup.select('ul > li')
        
        if not movie_list:
            print("영화 목록을 찾지 못했습니다.")
            return

        print("순수 영화 순위 추출을 시작합니다...\n")
        
        rank = 1 # 순위 초기화
        for movie in movie_list:
            # 1. 포스터 이미지가 있는지 '먼저' 확인
            img_elem = movie.select_one('img')
            
            # 💡 핵심 수정: 이미지가 없으면 메뉴(로그인, 예매 등)로 간주하고 과감히 패스!
            if not img_elem or 'src' not in img_elem.attrs:
                continue
                
            # 2. 영화 제목 추출
            title_elem = movie.select_one('.title') or movie.select_one('h3') or movie.select_one('a')
            if not title_elem:
                continue
                
            title = title_elem.text.strip()
            # 텍스트가 비어있어도 패스
            if not title:
                continue
                
            safe_title = "".join([c for c in title if c.isalnum() or c in (' ', '_')]).rstrip()
            
            # 3. 이미지 주소 합치기
            img_url = urljoin(url, img_elem['src'])
                
            # 4. 이미지 다운로드 및 로컬 저장
            img_path = os.path.join(folder_name, f"{rank}_{safe_title}.jpg")
            img_response = requests.get(img_url, headers=headers)
            
            if img_response.status_code == 200:
                with open(img_path, 'wb') as f:
                    f.write(img_response.content)
                print(f"{rank}위: {title} (이미지 다운로드 완료)")
            else:
                print(f"{rank}위: {title} (이미지 다운로드 실패)")
            
            # 💡 핵심 수정: 진짜 영화 데이터 처리가 끝났을 때만 순위(+1) 증가!
            rank += 1
            
    except Exception as e:
        print(f"오류 발생: {e}")

if __name__ == "__main__":
    scrape_movie_chart_requests()

페이지 정보를 가져오는 중입니다...
순수 영화 순위 추출을 시작합니다...

1위: 왕과 사는 남자 (이미지 다운로드 완료)
2위: 휴민트 (이미지 다운로드 완료)
3위: 초속 5센티미터 (이미지 다운로드 완료)
4위: 너자 2 (이미지 다운로드 완료)
5위: 슬라이드 스트럼 뮤트 (이미지 다운로드 완료)
6위: 넘버원 (이미지 다운로드 완료)
7위: 햄넷 (이미지 다운로드 완료)
8위: 부흥 (이미지 다운로드 완료)
9위: 매드 댄스 오피스 (이미지 다운로드 완료)
10위: 신의악단 (이미지 다운로드 완료)
11위: 렌탈 패밀리: 가족을 빌려드립니다 (이미지 다운로드 완료)
12위: 몬테크리스토 백작 (이미지 다운로드 완료)
13위: 호퍼스 (이미지 다운로드 완료)
14위: 만약에 우리 (이미지 다운로드 완료)
15위: 프랑켄슈타인 (이미지 다운로드 완료)
16위: 폭풍의 언덕 (이미지 다운로드 완료)
17위: 점보 (이미지 다운로드 완료)
18위: 아기 티라노 디보: 초식이지만 괜찮아! (이미지 다운로드 완료)
19위: 직사각형, 삼각형 (이미지 다운로드 완료)
20위: 아바타: 불과 재 (이미지 다운로드 완료)
